In [ ]:
from langchain_classic.prompts import ChatPromptTemplate,PromptTemplate
from langchain_classic.prompts import MessagesPlaceholder, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec,Pinecone,P
from dotenv import load_dotenv


In [14]:
load_dotenv()

True

In [17]:
pc=Pinecone(api_key="pcsk_6w7thy_3jDhyKLzVyFDNf3whdCfmH5ro533a85A49wWxi5N7jhsjSDJxito5uNgPreTtfS")

In [ ]:
index_name = "practice" 

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [19]:
file_path = "D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

In [21]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name=model_name)

C:\Users\hp\AppData\Local\Temp\ipykernel_6508\3298190680.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name=model_name)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 896.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
vectorstore = PineconeVectorStore.from_documents(
    texts,
    embedder,
    index_name=index_name
)

In [26]:
retriever = vectorstore.as_retriever(search_type="similarity")
query = "What is the course about?"
results = retriever.invoke(query)

In [ ]:
print(results)

[Document(id='1e8766e5-ab49-4f0b-982f-bd31b32bdc88', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, page_content='2. Course Objectives'), Document(id='378b76e6-c154-4c8a-9552-7c1d8ef0da94', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, page_content='This course aims to provide students with comprehensive theoretical and practical'), Document(id='dc250877-afad-4428-a316-455f3167a539', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creat

In [31]:
for i, doc in enumerate(results[:3], 1):
    print(f"{i}. {doc.page_content[:200]}")

1. 2. Course Objectives
2. This course aims to provide students with comprehensive theoretical and practical
3. 4. Course Content
